# 11 - TransXion and AMLNet Stress-Test Synthesis

Notebook này tổng hợp hậu nghiệm hai stress test đã khóa. Nó không train model, không audit
lại rule và không chọn lại policy. Hãy Add Input hai Kaggle version outputs của Notebook 09
và 10. Mọi artifact phải qua checksum lineage trước khi được đưa vào bảng khóa luận.

In [ ]:
from pathlib import Path
from importlib import metadata as importlib_metadata
import importlib
import importlib.util
import json
import os
import subprocess
import sys

from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
BRANCH = "main"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

def sync_project() -> Path:
    if KAGGLE:
        if not KAGGLE_PROJECT_DIR.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(KAGGLE_PROJECT_DIR)],
                check=True,
            )
        elif not (KAGGLE_PROJECT_DIR / ".git").is_dir():
            raise RuntimeError(f"Expected a Git clone at {KAGGLE_PROJECT_DIR}")
        else:
            subprocess.run(
                ["git", "-C", str(KAGGLE_PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
        return KAGGLE_PROJECT_DIR.resolve()
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook inside the project or on Kaggle with Internet enabled")

PROJECT_ROOT = sync_project()
while str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# A Kaggle kernel may survive a previous run. Never execute a stale src module.
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

QUICK_RUN = os.getenv("THESIS_STRESS_QUICK_RUN", "0") == "1"
USE_TEST_FIXTURE = os.getenv("THESIS_STRESS_TEST_FIXTURE", "0") == "1"
if USE_TEST_FIXTURE and not QUICK_RUN:
    raise ValueError("THESIS_STRESS_TEST_FIXTURE=1 requires THESIS_STRESS_QUICK_RUN=1")
STRICT_RUNTIME_REQUIREMENTS = not (QUICK_RUN and USE_TEST_FIXTURE)

OUTPUT_BASE = (
    Path("/kaggle/working/thesis_stress_outputs")
    if KAGGLE else PROJECT_ROOT / "results/stress/notebooks"
)
DATA_ROOTS = [Path("/kaggle/input")] if KAGGLE else [PROJECT_ROOT / "data/raw"]

RUNTIME_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scikit-learn": "sklearn",
    "scipy": "scipy",
    "pyyaml": "yaml",
    "joblib": "joblib",
    "torch": "torch",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
}

declared_requirements = {}
for raw_line in (PROJECT_ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    requirement_text = raw_line.split("#", maxsplit=1)[0].strip()
    if not requirement_text:
        continue
    requirement = Requirement(requirement_text)
    normalized_name = canonicalize_name(requirement.name)
    if normalized_name in RUNTIME_IMPORTS:
        declared_requirements[normalized_name] = requirement
missing_declarations = sorted(set(RUNTIME_IMPORTS).difference(declared_requirements))
if missing_declarations:
    raise RuntimeError(
        "requirements.txt has no declared range for runtime packages: "
        + ", ".join(missing_declarations)
    )

def runtime_package_status(package_name):
    requirement = declared_requirements[package_name]
    import_name = RUNTIME_IMPORTS[package_name]
    if importlib.util.find_spec(import_name) is None:
        return "missing", None, requirement
    try:
        installed_version = importlib_metadata.version(requirement.name)
    except importlib_metadata.PackageNotFoundError:
        installed_version = None
    # Vendor-managed Torch images can expose a fully usable import while the
    # distribution metadata has no Version field. Torch is never repaired in
    # this notebook, so reading its runtime version cannot create a stale
    # import after pip installation.
    if not installed_version and package_name == "torch":
        torch_module = importlib.import_module(import_name)
        installed_version = str(getattr(torch_module, "__version__", "")).strip() or None
    if not installed_version:
        return "metadata_missing", None, requirement
    if installed_version not in requirement.specifier:
        return "out_of_range", installed_version, requirement
    return "ok", installed_version, requirement

runtime_status = {
    package_name: runtime_package_status(package_name)
    for package_name in RUNTIME_IMPORTS
}
torch_status, torch_version, torch_requirement = runtime_status["torch"]
if torch_status != "ok" and STRICT_RUNTIME_REQUIREMENTS:
    observed = torch_version if torch_version is not None else torch_status
    raise ImportError(
        f"PyTorch environment is incompatible: observed {observed!r}, required "
        f"{torch_requirement}. Select a Kaggle image whose preinstalled Torch/CUDA wheel "
        "satisfies this range; this notebook will not install, upgrade, downgrade, or replace Torch."
    )

repair_requirements = [
    str(requirement)
    for package_name, (status, _, requirement) in runtime_status.items()
    if package_name != "torch" and status != "ok"
]
if repair_requirements and KAGGLE and STRICT_RUNTIME_REQUIREMENTS:
    # --no-deps ensures this repair cannot replace Kaggle's preinstalled
    # Torch/CUDA wheel indirectly through dependency resolution.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            "--no-deps",
            *repair_requirements,
        ],
        check=True,
    )
    importlib.invalidate_caches()
    runtime_status = {
        package_name: runtime_package_status(package_name)
        for package_name in RUNTIME_IMPORTS
    }

invalid_packages = {
    package_name: {
        "status": status,
        "installed_version": installed_version,
        "required": str(requirement),
    }
    for package_name, (status, installed_version, requirement) in runtime_status.items()
    if status != "ok"
}
RUNTIME_REQUIREMENTS_VALID = not invalid_packages
if invalid_packages and STRICT_RUNTIME_REQUIREMENTS:
    raise ImportError(
        "Runtime packages do not satisfy requirements.txt: "
        + json.dumps(invalid_packages, sort_keys=True)
    )
RUNTIME_PACKAGE_VERSIONS = {
    declared_requirements[package_name].name: installed_version
    for package_name, (_, installed_version, _) in runtime_status.items()
}
if invalid_packages and not STRICT_RUNTIME_REQUIREMENTS:
    print(
        "Engineering fixture warning: runtime packages are outside the declared "
        "ranges. This run is claim-ineligible by construction: "
        + json.dumps(invalid_packages, sort_keys=True)
    )

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "test_fixture": USE_TEST_FIXTURE,
    "claim_eligible_mode": not QUICK_RUN and not USE_TEST_FIXTURE,
    "strict_runtime_requirements": STRICT_RUNTIME_REQUIREMENTS,
    "runtime_requirements_valid": RUNTIME_REQUIREMENTS_VALID,
    "runtime_package_versions": RUNTIME_PACKAGE_VERSIONS,
})

## 1. Discover exactly one valid artifact for each stress dataset

In [ ]:
from src.stress_testing.artifacts import find_unique_stress_manifest, validate_stress_lineage

SEARCH_ROOTS = [Path("/kaggle/input")] if KAGGLE else [OUTPUT_BASE]
manifest_paths = {
    "transxion_v2": find_unique_stress_manifest("transxion_v2", SEARCH_ROOTS),
    "amlnet_v1_0": find_unique_stress_manifest("amlnet_v1_0", SEARCH_ROOTS),
}
manifests = {}
for dataset, manifest_path in manifest_paths.items():
    lineage_path = manifest_path.with_name("stress_lineage.json")
    validate_stress_lineage(lineage_path, expected_dataset_manifest=manifest_path)
    manifests[dataset] = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(dataset, manifest_path, "lineage=valid")

## 2. Claim eligibility and source identity gate

In [ ]:
CORE_ENVIRONMENT_PACKAGES = (
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "xgboost",
    "lightgbm",
    "torch",
)

def manifest_core_environment(payload):
    environment = payload.get("environment")
    if not isinstance(environment, dict):
        raise ValueError("Stress manifest is missing its captured environment")
    python_runtime = str(environment.get("python") or "").strip()
    packages = environment.get("packages")
    if not python_runtime or not isinstance(packages, dict):
        raise ValueError(
            "Stress manifest must record Python and core package versions"
        )
    signature = {"python": python_runtime.split()[0]}
    for package_name in CORE_ENVIRONMENT_PACKAGES:
        version = packages.get(package_name)
        if version is None or not str(version).strip():
            raise ValueError(
                f"Stress manifest is missing version metadata for {package_name}"
            )
        signature[package_name] = str(version).strip()
    return signature

upstream_core_environments = {
    dataset: manifest_core_environment(payload)
    for dataset, payload in manifests.items()
}
serialized_environments = {
    json.dumps(environment, sort_keys=True)
    for environment in upstream_core_environments.values()
}
if len(serialized_environments) != 1:
    raise ValueError(
        "Notebook 09/10 artifacts must use matching Python and core package versions: "
        + json.dumps(upstream_core_environments, sort_keys=True)
    )
MATCHED_UPSTREAM_CORE_ENVIRONMENT = next(
    iter(upstream_core_environments.values())
)
CURRENT_CORE_ENVIRONMENT = {
    "python": sys.version.split()[0],
    **{
        package_name: RUNTIME_PACKAGE_VERSIONS[package_name]
        for package_name in CORE_ENVIRONMENT_PACKAGES
    },
}

eligibility = pd.DataFrame([
    {
        "dataset": dataset,
        "claim_eligible": payload["claim_eligible"],
        "claim_blockers": ";".join(payload["claim_blockers"]),
        "git_commit": payload["git_commit"],
        "pipeline_fingerprint": payload["stress_pipeline_fingerprint"],
        "source_fingerprint": payload["stress_source_fingerprint"],
        "core_environment": json.dumps(
            upstream_core_environments[dataset], sort_keys=True
        ),
        "source_checksum_verified": all(
            record.get("verified", False)
            for record in payload["data_manifest"]["source_files"]
            if record.get("role", "transactions") != "profile"
        ),
    }
    for dataset, payload in manifests.items()
])
display(eligibility)
if not eligibility["claim_eligible"].all() and not (QUICK_RUN and USE_TEST_FIXTURE):
    raise ValueError("At least one stress artifact is not eligible for thesis claims")
if not eligibility["claim_eligible"].all():
    display(Markdown(
        "**Engineering smoke only:** upstream fixture/quick artifacts are intentionally "
        "ineligible and this synthesis must not be used in the thesis results."
    ))
upstream_commits = eligibility["git_commit"].fillna("").astype(str).str.strip()
if upstream_commits.eq("").any():
    raise ValueError(
        "Every Notebook 09/10 artifact must record a non-null Git commit"
    )
source_fingerprints = set(
    eligibility["source_fingerprint"].fillna("").astype(str).str.strip()
)
source_fingerprints.discard("")
if len(source_fingerprints) != 1:
    raise ValueError(
        "Notebook 09/10 artifacts must use one non-null executable source fingerprint"
    )
MATCHED_STRESS_SOURCE_FINGERPRINT = next(iter(source_fingerprints))

## 3. Read locked outputs without re-selection

In [ ]:
def read_table(dataset, filename):
    return pd.read_csv(manifest_paths[dataset].parent / filename).assign(dataset=dataset)

def read_json_artifact(dataset, filename):
    return json.loads(
        (manifest_paths[dataset].parent / filename).read_text(encoding="utf-8")
    )

coverage = pd.concat(
    [read_table(dataset, "coverage_results.csv") for dataset in manifest_paths],
    ignore_index=True,
)
predictive = pd.concat(
    [read_table(dataset, "predictive_metrics.csv") for dataset in manifest_paths],
    ignore_index=True,
)
matched = pd.concat(
    [read_table(dataset, "matched_risk_results.csv") for dataset in manifest_paths],
    ignore_index=True,
)
bootstrap = pd.concat(
    [read_table(dataset, "paired_bootstrap.csv") for dataset in manifest_paths],
    ignore_index=True,
)
controls = pd.concat(
    [read_table(dataset, "negative_controls.csv") for dataset in manifest_paths],
    ignore_index=True,
)
outcomes = pd.concat(
    [read_table(dataset, "stress_outcomes.csv") for dataset in manifest_paths],
    ignore_index=True,
)
residual = pd.concat(
    [read_table(dataset, "residual_evidence.csv") for dataset in manifest_paths],
    ignore_index=True,
)
ablations = pd.concat(
    [read_table(dataset, "ablation_results.csv") for dataset in manifest_paths],
    ignore_index=True,
)
predictor_sensitivity = pd.concat(
    [read_table(dataset, "predictor_explanation_sensitivity.csv") for dataset in manifest_paths],
    ignore_index=True,
)
contrastive_status = pd.DataFrame([
    {
        "dataset": dataset,
        "contrastive_meta_available": bool(
            read_json_artifact(dataset, "contrastive_meta_provenance.json")["available"]
        ),
        "contrastive_meta_fallback_reason": read_json_artifact(
            dataset, "contrastive_meta_provenance.json"
        ).get("fallback_reason"),
        "contrastive_meta_fit_partition": read_json_artifact(
            dataset, "contrastive_meta_provenance.json"
        )["fit_partition"],
    }
    for dataset in manifest_paths
])
display(predictive)
display(contrastive_status)

## 4. Primary fixed-coverage comparison

In [ ]:
primary = coverage[coverage["coverage_budget"].round(2).isin([0.10, 0.25])].copy()
primary = primary.sort_values(["dataset", "coverage_budget"]).reset_index(drop=True)
compact_columns = [
    "dataset", "predictor", "coverage_budget", "alert_count",
    "requested_selected_count", "realized_selected_count", "abstain",
    "abstention_reason", "all_alert_precision",
    "score_only_requested_budget_precision", "supported_alert_rate",
    "unsupported_alert_rate", "requested_explanation_coverage",
    "realized_explanation_coverage",
]
compact_summary = primary[compact_columns].copy()
display(compact_summary)

case_labels = [
    f"{dataset.replace('_', ' ').title()}\n{coverage:.0%}"
    for dataset, coverage in zip(
        compact_summary["dataset"], compact_summary["coverage_budget"]
    )
]
x = np.arange(len(compact_summary))
colors = ["#4C72B0" if dataset == "transxion_v2" else "#DD8452"
          for dataset in compact_summary["dataset"]]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
score_bars = axes[0, 0].bar(
    x, compact_summary["score_only_requested_budget_precision"], color=colors
)
axes[0, 0].bar_label(score_bars, fmt="%.3f", padding=3)
axes[0, 0].set(
    title="Score-only precision at requested budgets",
    ylabel="Precision", xticks=x, xticklabels=case_labels, ylim=(0, 1.10),
)

support = compact_summary.drop_duplicates("dataset")
support_labels = support["dataset"].str.replace("_", " ").str.title()
support_x = np.arange(len(support))
axes[0, 1].bar(
    support_x, support["supported_alert_rate"], label="Supported", color="#55A868"
)
axes[0, 1].bar(
    support_x, support["unsupported_alert_rate"],
    bottom=support["supported_alert_rate"], label="Unsupported", color="#C44E52"
)
axes[0, 1].set(
    title="Selected-rule support within predictor alerts",
    ylabel="Alert fraction", xticks=support_x, xticklabels=support_labels, ylim=(0, 1.05),
)
axes[0, 1].legend(loc="upper right")

width = 0.38
axes[1, 0].bar(
    x - width / 2, compact_summary["requested_explanation_coverage"],
    width, label="Requested", color="#8172B2"
)
realized_bars = axes[1, 0].bar(
    x + width / 2, compact_summary["realized_explanation_coverage"],
    width, label="Realized", color="#64B5CD"
)
axes[1, 0].bar_label(realized_bars, fmt="%.2f", padding=3)
axes[1, 0].set(
    title="Requested versus realized explanation coverage",
    ylabel="Coverage", xticks=x, xticklabels=case_labels, ylim=(0, 0.32),
)
axes[1, 0].legend(loc="upper left")

abstention_bars = axes[1, 1].bar(
    x, compact_summary["abstain"].astype(float), color=colors
)
axes[1, 1].bar_label(abstention_bars, fmt="%.0f", padding=3)
axes[1, 1].set(
    title="Policy-level guardrail abstention",
    ylabel="Abstention indicator", xticks=x, xticklabels=case_labels, ylim=(0, 1.10),
)
for axis in axes.flat:
    axis.grid(axis="y", alpha=0.3)

SYNTHESIS_DIR = OUTPUT_BASE / "11_stress_test_synthesis"
SYNTHESIS_DIR.mkdir(parents=True, exist_ok=True)
summary_figure_path = SYNTHESIS_DIR / "stress_test_summary.png"
fig.savefig(summary_figure_path, dpi=180, bbox_inches="tight")
plt.show()

## 5. Uncertainty, matched-risk and negative-control evidence

In [ ]:
display(bootstrap[bootstrap["coverage"].round(2).isin([0.10, 0.25])])
display(matched[matched["coverage_budget"].round(2).isin([0.10, 0.25])])
display(controls)
display(outcomes[outcomes["coverage_budget"].round(2).isin([0.10, 0.25])])
display(ablations[ablations["coverage_budget"].round(2).isin([0.10, 0.25])])
display(predictor_sensitivity[
    predictor_sensitivity["coverage_budget"].round(2).isin([0.10, 0.25])
])

bootstrap_primary = bootstrap[
    bootstrap["coverage"].round(2).isin([0.10, 0.25])
][[
    "dataset", "coverage", "ci_low", "ci_high", "valid_iterations",
    "probability_vasre_better", "outcome"
]].rename(columns={"outcome": "bootstrap_outcome"})
matched_primary = matched[
    matched["coverage_budget"].round(2).isin([0.10, 0.25])
][[
    "dataset", "coverage_budget", "matched_pairs", "match_rate",
    "mean_absolute_risk_gap", "matched_precision_difference",
    "matched_ci_low", "matched_ci_high"
]]
outcome_columns = [
    "dataset", "coverage_budget", "outcome", "risk_conditioned_tp_fp_gap",
    "positive_incremental_evidence_confirmed", "saturation_confirmed",
    "saturation_scope_eligible", "saturation_sample_adequate",
    "core_equal_count_score_only_comparison",
    "shuffled_control_comparison_valid",
    "shuffled_controls_below_locked_primary",
    "shuffled_controls_no_material_gain"
]
outcome_primary = outcomes[
    outcomes["coverage_budget"].round(2).isin([0.10, 0.25])
][[column for column in outcome_columns if column in outcomes]].rename(
    columns={"outcome": "stress_outcome"}
)
synthesis = primary.merge(
    bootstrap_primary,
    left_on=["dataset", "coverage_budget"],
    right_on=["dataset", "coverage"],
    how="left",
    validate="one_to_one",
).merge(
    matched_primary,
    on=["dataset", "coverage_budget"],
    how="left",
    validate="one_to_one",
).merge(
    outcome_primary,
    on=["dataset", "coverage_budget"],
    how="left",
    validate="one_to_one",
).merge(
    contrastive_status,
    on="dataset",
    how="left",
    validate="many_to_one",
)
display(synthesis)

## 6. Export a checksum-ready synthesis table

In [ ]:
from datetime import datetime, timezone
import hashlib

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SYNTHESIS_DIR = OUTPUT_BASE / "11_stress_test_synthesis"
SYNTHESIS_DIR.mkdir(parents=True, exist_ok=True)
synthesis_path = SYNTHESIS_DIR / "primary_stress_test_synthesis.csv"
compact_summary_path = SYNTHESIS_DIR / "primary_stress_test_summary.csv"
eligibility_path = SYNTHESIS_DIR / "stress_input_eligibility.csv"
synthesis.to_csv(synthesis_path, index=False)
compact_summary.to_csv(compact_summary_path, index=False)
eligibility.to_csv(eligibility_path, index=False)
upstream = {
    dataset: {
        "stress_test_manifest": str(manifest_path),
        "stress_test_manifest_sha256": sha256_path(manifest_path),
        "stress_lineage_sha256": sha256_path(manifest_path.with_name("stress_lineage.json")),
        "git_commit": manifests[dataset]["git_commit"],
        "pipeline_fingerprint": manifests[dataset]["stress_pipeline_fingerprint"],
        "source_fingerprint": manifests[dataset]["stress_source_fingerprint"],
        "core_environment": upstream_core_environments[dataset],
    }
    for dataset, manifest_path in manifest_paths.items()
}
synthesis_note = {
    "schema_version": 1,
    "analysis_type": "post_hoc_locked_artifact_synthesis",
    "datasets": sorted(manifest_paths),
    "primary_coverages": [0.10, 0.25],
    "model_rule_or_policy_reselection": False,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": GIT_COMMIT,
    "source_fingerprint": MATCHED_STRESS_SOURCE_FINGERPRINT,
    "upstream_core_environment_match_required": True,
    "matched_upstream_core_environment": MATCHED_UPSTREAM_CORE_ENVIRONMENT,
    "synthesis_runtime_core_environment": CURRENT_CORE_ENVIRONMENT,
    "upstream": upstream,
    "outputs": {
        synthesis_path.name: sha256_path(synthesis_path),
        compact_summary_path.name: sha256_path(compact_summary_path),
        eligibility_path.name: sha256_path(eligibility_path),
        summary_figure_path.name: sha256_path(summary_figure_path),
    },
    "claim_boundary": "Stress evidence does not establish causality or production generalization.",
}
synthesis_manifest_path = SYNTHESIS_DIR / "synthesis_manifest.json"
synthesis_manifest_path.write_text(
    json.dumps(synthesis_note, indent=2), encoding="utf-8"
)
synthesis_lineage = {
    "schema_version": 1,
    "manifest": synthesis_manifest_path.name,
    "manifest_sha256": sha256_path(synthesis_manifest_path),
    "outputs": synthesis_note["outputs"],
    "upstream_manifest_sha256": {
        dataset: values["stress_test_manifest_sha256"]
        for dataset, values in upstream.items()
    },
    "matched_upstream_core_environment": MATCHED_UPSTREAM_CORE_ENVIRONMENT,
    "synthesis_runtime_core_environment": CURRENT_CORE_ENVIRONMENT,
}
synthesis_lineage_path = SYNTHESIS_DIR / "synthesis_lineage.json"
synthesis_lineage_path.write_text(
    json.dumps(synthesis_lineage, indent=2), encoding="utf-8"
)
assert sha256_path(synthesis_manifest_path) == synthesis_lineage["manifest_sha256"]
print({
    "output_dir": str(SYNTHESIS_DIR),
    "rows": len(synthesis),
    "manifest_sha256": synthesis_lineage["manifest_sha256"],
})

In [ ]:
reference_predictive = predictive[predictive["reference_selected"]].set_index("dataset")
dataset_labels = {
    "transxion_v2": "TransXion v2",
    "amlnet_v1_0": "AMLNet v1.0",
}
dataset_findings = []
for dataset in ("transxion_v2", "amlnet_v1_0"):
    rows = compact_summary[compact_summary["dataset"] == dataset].sort_values(
        "coverage_budget"
    )
    precision_by_budget = ", ".join(
        f"{row.coverage_budget:.0%}: {row.score_only_requested_budget_precision:.3f}"
        for row in rows.itertuples()
    )
    predictor_row = reference_predictive.loc[dataset]
    dataset_findings.append(
        f"- **{dataset_labels[dataset]}:** reference {predictor_row['family']} đạt "
        f"PR-AUC {predictor_row['pr_auc']:.3f}; selected-rule support trên alerts là "
        f"{rows['supported_alert_rate'].iloc[0]:.1%}; score-only precision theo budget "
        f"({precision_by_budget})."
    )

all_primary_abstained = bool(compact_summary["abstain"].all())
positive_incremental_evidence = bool(
    synthesis["positive_incremental_evidence_confirmed"].fillna(False).any()
)
saturation_confirmed = bool(synthesis["saturation_confirmed"].fillna(False).any())
decision_text = (
    "Cả hai stress dataset đều kích hoạt guardrail abstention tại coverage 10% và 25%."
    if all_primary_abstained else
    "Ít nhất một stress dataset phát hành selective explanations tại primary coverage."
)
evidence_text = (
    "Có bằng chứng incremental dương đã qua confirmatory gate."
    if positive_incremental_evidence else
    "Không có bằng chứng confirmatory rằng rule evidence cải thiện predictor score."
)
saturation_text = (
    "Saturation được xác nhận theo protocol đã khóa."
    if saturation_confirmed else
    "Saturation không được xác nhận; không suy diễn từ zero-selection hoặc bootstrap không khả dụng."
)
display(Markdown(
    "## 7. Thesis interpretation\n\n"
    + "\n".join(dataset_findings)
    + "\n\n"
    + f"- {decision_text}\n"
    + f"- {evidence_text}\n"
    + f"- {saturation_text}\n"
    + "- Đây là stress-test robustness trong hai benchmark nghiên cứu; kết quả không "
      "chứng minh causal explanation, external validation thực tế hoặc production generalization."
))